<div style="border-left:4px solid #3778b7;background-color:rgba(55,120,183,0.08);padding:18px 20px;margin:8px 0 14px 0;border-radius:4px;font-size:16px;line-height:1.6;color:inherit;">

# <span style="color:#3778b7;font-size:26px;font-weight:600;line-height:1.4;">Actividad 4. Solución analítica de Black–Scholes</span>

Se siguen los cinco pasos de la actividad y se calcula el precio de la
opción de compra de IBM. Cada bloque se ejecuta después del anterior.

Se parte de
$$
V_t+\frac{\sigma^2S^2}{2}V_{SS}+(r-q)SV_S-rV=0,
\qquad V(S,T)=(S-E)^+.
$$

**Preparación.** SymPy permite derivar y simplificar las expresiones.
Las letras se conservan como símbolos hasta sustituir los datos de IBM.

</div>

In [1]:
import sympy as sp
from IPython.display import display

# Los precios, la volatilidad y los tiempos son positivos.
S, E, sigma, T, tau = sp.symbols("S E sigma T tau", positive=True)
t, x, s = sp.symbols("t x s", real=True)

# La variable tasa representa r. Se conserva el nombre usado en clase.
tasa, q, alpha, beta = sp.symbols("r q alpha beta", real=True)


<div style="border-left:3px solid #3778b7;background-color:rgba(55,120,183,0.035);padding:12px 16px;margin:8px 0 14px 0;border-radius:4px;font-size:16px;line-height:1.6;color:inherit;">

## <span style="color:#3778b7;font-size:22px;font-weight:600;line-height:1.4;">1. Cambio de tiempo: </span>$\tau=T-t$

Se escribe $V(S,t)=W(S,T-t)$. La derivada respecto de $t$ cambia de
signo: $V_t=-W_\tau$. SymPy calcula las derivadas con `sp.diff`.
En las salidas, `display` presenta las expresiones como fórmulas.

</div>

In [2]:
W = sp.Function("W")
V_tiempo = W(S, T - t)

# Se calcula cada derivada por separado.
V_t = sp.diff(V_tiempo, t)
V_S = sp.diff(V_tiempo, S)
V_SS = sp.diff(V_tiempo, S, 2)

ecuacion_V = V_t + sigma**2 * S**2 / 2 * V_SS
ecuacion_V = ecuacion_V + (tasa - q) * S * V_S - tasa * V_tiempo

# subs reemplaza t por T-tau; doit termina las derivadas pendientes.
ecuacion_W = ecuacion_V.subs(t, T - tau)
ecuacion_W = -ecuacion_W.doit()  # Se multiplica por -1.
ecuacion_W = sp.simplify(ecuacion_W)

display(sp.Eq(ecuacion_W, 0))
display(sp.Eq(W(S, 0), sp.Max(S - E, 0)))


Eq(-S**2*sigma**2*Derivative(W(S, tau), (S, 2))/2 + S*(q - r)*Derivative(W(S, tau), S) + r*W(S, tau) + Derivative(W(S, tau), tau), 0)

Eq(W(S, 0), Max(0, -E + S))

<div style="border-left:3px solid #3778b7;background-color:rgba(55,120,183,0.035);padding:12px 16px;margin:8px 0 14px 0;border-radius:4px;font-size:16px;line-height:1.6;color:inherit;">

## <span style="color:#3778b7;font-size:22px;font-weight:600;line-height:1.4;">2. Cambio de precio: </span>$x=\ln S$

Se escribe $W(S,\tau)=Z(\ln S,\tau)$. Por la regla de la cadena,
$$W_S=\frac{Z_x}{S},\qquad
W_{SS}=\frac{Z_{xx}-Z_x}{S^2}.$$
Se calculan estas derivadas y se sustituyen en la ecuación del paso 1.

</div>

In [3]:
Z = sp.Function("Z")
W_log = Z(sp.log(S), tau)

W_tau = sp.diff(W_log, tau)
W_S = sp.diff(W_log, S)
W_SS = sp.diff(W_log, S, 2)

# Es la ecuación del paso 1, escrita con las nuevas derivadas.
ecuacion_log = W_tau - sigma**2 * S**2 / 2 * W_SS
ecuacion_log = ecuacion_log - (tasa - q) * S * W_S + tasa * W_log

# Se vuelve a escribir log(S) como x y se simplifica.
ecuacion_Z = ecuacion_log.subs(sp.log(S), x)
ecuacion_Z = ecuacion_Z.doit()
ecuacion_Z = sp.simplify(ecuacion_Z)

display(sp.Eq(ecuacion_Z, 0))
display(sp.Eq(Z(x, 0), sp.Max(sp.exp(x) - E, 0)))


Eq(r*Z(x, tau) + sigma**2*(Derivative(Z(x, tau), x) - Derivative(Z(x, tau), (x, 2)))/2 + (q - r)*Derivative(Z(x, tau), x) + Derivative(Z(x, tau), tau), 0)

Eq(Z(x, 0), Max(0, -E + exp(x)))

<div style="border-left:3px solid #3778b7;background-color:rgba(55,120,183,0.035);padding:12px 16px;margin:8px 0 14px 0;border-radius:4px;font-size:16px;line-height:1.6;color:inherit;">

## <span style="color:#3778b7;font-size:22px;font-weight:600;line-height:1.4;">3. Transformación exponencial</span>

Se usa la misma convención en todo el cuaderno:
$$Z(x,\tau)=e^{-\alpha x-\beta\tau}u(x,\tau).$$
Los valores que eliminan los términos en $u_x$ y $u$ son
$$\alpha=\frac{r-q-\sigma^2/2}{\sigma^2},\qquad
\beta=r+\frac{\sigma^2\alpha^2}{2}.$$
El código calcula las derivadas, sustituye estos valores y muestra
la ecuación del calor. Su condición inicial es
$u(x,0)=e^{\alpha x}(e^x-E)^+$.

</div>

In [4]:
u = sp.Function("u")(x, tau)
factor = sp.exp(-alpha * x - beta * tau)
Z_exp = factor * u

Z_tau = sp.diff(Z_exp, tau)
Z_x = sp.diff(Z_exp, x)
Z_xx = sp.diff(Z_exp, x, 2)

ecuacion_u = Z_tau - sigma**2 / 2 * Z_xx
ecuacion_u = ecuacion_u - (tasa - q - sigma**2 / 2) * Z_x
ecuacion_u = ecuacion_u + tasa * Z_exp

# Se divide entre la exponencial común y se agrupan los términos.
ecuacion_u = sp.expand(ecuacion_u / factor)

# Estos valores hacen cero los coeficientes de u_x y de u.
alpha_valor = (tasa - q - sigma**2 / 2) / sigma**2
beta_valor = tasa + sigma**2 * alpha_valor**2 / 2
display(sp.Eq(alpha, alpha_valor))
display(sp.Eq(beta, beta_valor))

ecuacion_calor = ecuacion_u.subs(alpha, alpha_valor)
ecuacion_calor = ecuacion_calor.subs(beta, beta_valor)
ecuacion_calor = sp.simplify(ecuacion_calor)
display(sp.Eq(ecuacion_calor, 0))


Eq(alpha, (-q + r - sigma**2/2)/sigma**2)

Eq(beta, r + (-q + r - sigma**2/2)**2/(2*sigma**2))

Eq(-sigma**2*Derivative(u(x, tau), (x, 2))/2 + Derivative(u(x, tau), tau), 0)

<div style="border-left:3px solid #3778b7;background-color:rgba(55,120,183,0.035);padding:12px 16px;margin:8px 0 14px 0;border-radius:4px;font-size:16px;line-height:1.6;color:inherit;">

## <span style="color:#3778b7;font-size:22px;font-weight:600;line-height:1.4;">4. Fórmula de Green</span>

Se utiliza el núcleo dado en la actividad. Para una opción de compra,
el pago es $e^s-E$ cuando $s>\ln E$ y cero en el resto. Por eso la
integral para $V$ comienza en $\ln E$.

</div>

In [5]:
# G es el factor que multiplica la condición inicial en la integral.
G = sp.exp(-(x - s)**2 / (2 * sigma**2 * tau))
G = G / sp.sqrt(2 * sp.pi * sigma**2 * tau)

# Un resultado igual a cero comprueba la ecuación del calor para G.
residuo_G = sp.diff(G, tau) - sigma**2 / 2 * sp.diff(G, x, 2)
print("Comprobación del núcleo de Green:", sp.simplify(residuo_G))

# Se regresa a S mediante x=log(S) y se incorpora el pago de la opción.
G_precio = G.subs(x, sp.log(S))
pago = sp.exp(s) - E
integrando = G_precio * sp.exp(alpha * s) * pago
integral = sp.Integral(integrando, (s, sp.log(E), sp.oo))
V_green = sp.exp(-beta * tau) * S**(-alpha) * integral
display(V_green)


Comprobación del núcleo de Green: 0


exp(-beta*tau)*Integral(sqrt(2)*(-E + exp(s))*exp(alpha*s)*exp(-(-s + log(S))**2/(2*sigma**2*tau))/(2*sqrt(pi)*sigma*sqrt(tau)), (s, log(E), oo))/S**alpha

<div style="border-left:4px solid #27846f;background-color:rgba(39,132,111,0.07);padding:16px 20px;margin:8px 0 14px 0;border-radius:4px;font-size:16px;line-height:1.6;color:inherit;">

Al hacer el cambio $y=s-\ln S$ y completar el cuadrado, la integral
se expresa con la función de distribución normal estándar $N$:
$$V=Se^{-q\tau}N(d_1)-Ee^{-r\tau}N(d_2).$$
La función error `sp.erf` permite calcular $N$ en SymPy.

</div>

In [6]:
def N(z):
    """Probabilidad de que una normal estándar tome un valor menor que z."""
    return (1 + sp.erf(z / sp.sqrt(2))) / 2


d_1 = (sp.log(S / E) + (tasa - q + sigma**2 / 2) * tau)
d_1 = d_1 / (sigma * sp.sqrt(tau))
d_2 = d_1 - sigma * sp.sqrt(tau)

V = S * sp.exp(-q * tau) * N(d_1)
V = V - E * sp.exp(-tasa * tau) * N(d_2)
display(sp.Eq(sp.Function("V")(S, T - tau), V))


Eq(V(S, T - tau), -E*(erf(sqrt(2)*(-sigma*sqrt(tau) + (tau*(-q + r + sigma**2/2) + log(S/E))/(sigma*sqrt(tau)))/2)/2 + 1/2)*exp(-r*tau) + S*(erf(sqrt(2)*(tau*(-q + r + sigma**2/2) + log(S/E))/(2*sigma*sqrt(tau)))/2 + 1/2)*exp(-q*tau))

<div style="border-left:3px solid #27846f;background-color:rgba(39,132,111,0.045);padding:12px 16px;margin:8px 0 14px 0;border-radius:4px;font-size:16px;line-height:1.6;color:inherit;">

## <span style="color:#27846f;font-size:22px;font-weight:600;line-height:1.4;">5. Precio de la opción de IBM</span>

Se usan $S=58.5$, $E=60$, $\sigma=0.29$, $r=0.04$ y
$\tau=T-t=0.3-0=0.3$ años. Como la acción no paga dividendos, $q=0$.
El diccionario `datos` indica qué número sustituye a cada símbolo.
`sp.N` convierte el resultado a una aproximación decimal.

</div>

In [7]:
datos = {
    S: 58.5,        # Precio actual de la acción.
    E: 60,          # Precio de ejercicio.
    sigma: 0.29,    # Volatilidad anual.
    tasa: 0.04,     # Tasa de interés anual.
    q: 0,           # La acción no paga dividendos.
    tau: 0.3,       # Años que faltan para el vencimiento.
}

d1_IBM = d_1.subs(datos)
d2_IBM = d_2.subs(datos)
precio = V.subs(datos)

print("d1 =", sp.N(d1_IBM, 10))
print("d2 =", sp.N(d2_IBM, 10))
print("N(d1) =", sp.N(N(d1_IBM), 10))
print("N(d2) =", sp.N(N(d2_IBM), 10))
print("V(58.5, 0) =", sp.N(precio, 12))
print(f"Precio de la opción: {float(precio):.2f} unidades monetarias")


d1 = -0.004424641225
d2 = -0.1632641829
N(d1) = 0.4982348293
N(d2) = 0.4351552167
V(58.5, 0) = 3.34886389501
Precio de la opción: 3.35 unidades monetarias


<div style="border-left:3px solid #8064b0;background-color:rgba(128,100,176,0.06);padding:12px 16px;margin:8px 0 14px 0;border-radius:4px;font-size:16px;line-height:1.6;color:inherit;">

<span style="color:#8064b0;font-size:19px;font-weight:600;">**Comprobación con la integral.**</span> Se sustituyen los mismos datos en
la integral de Green del paso 4. Ambos cálculos deben dar el mismo
precio, salvo diferencias de redondeo.

</div>

In [8]:
# Primero se sustituyen alpha y beta; después, los datos de IBM.
V_green_IBM = V_green.subs(alpha, alpha_valor)
V_green_IBM = V_green_IBM.subs(beta, beta_valor)
V_green_IBM = V_green_IBM.subs(datos)

precio_green = sp.N(V_green_IBM, 12)
precio_formula = sp.N(precio, 12)
diferencia = abs(precio_green - precio_formula)

print("Precio con Green:", precio_green)
print("Precio con Black–Scholes:", precio_formula)
print("Diferencia absoluta:", diferencia)


Precio con Green: 3.34886389501
Precio con Black–Scholes: 3.34886389501
Diferencia absoluta: 0
